# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata (this builds the metadata representation, not the entire data)
dataset = mlc.Dataset(croissant_url)

# Access some key metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {meta.keywords}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column in Croissant is uniquely identified by its `@id`. We'll enumerate these to plan our data extraction steps.

In [ ]:
# List record sets available in this dataset, by @id and their corresponding fields
print("Available record sets (by @id and name):")
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<no name>')}")
    # List the fields
    print("  Fields in this record set (by @id):")
    for field in rs['fields']:
        # Each field here should be a dictionary
        print(f"    - Field @id: {field['@id']} Name: {field.get('name', field['@id'])}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. 

For this notebook, we extract data from all detected record sets. Each DataFrame column is referenced by its field or column `@id`.

In [ ]:
# Prepare the set of record set @ids for loading
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded dataframe with {len(df)} rows and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}: {e}")

if dataframes:
    # Pick the first available record set as default for demo and set variable names
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in selected record set ({selected_record_set_id}):")
    print(dataframes[selected_record_set_id].columns.tolist())
    print("\nFirst few rows:")
    display(dataframes[selected_record_set_id].head())
else:
    print('No record sets could be loaded!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate these steps using numeric and groupable fields by their `@id`. Adjust `numeric_field_id` and `group_field_id` to match your record set.


In [ ]:
# Example: Choose a numeric field and a groupable field (by their Croissant @id).
# You can adapt these IDs based on the print out in the overview step above.

# For demo purposes, try to auto-detect numeric fields
import numpy as np
df = dataframes[selected_record_set_id]

# Attempt to discover a numeric field (float/int type)
numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to cast columns to float, if possible
    for col in df.columns:
        try:
            df[col+'_float'] = pd.to_numeric(df[col], errors='coerce')
            if df[col+'_float'].notnull().sum() > 0:
                numeric_field_id = col+'_float'
                break
        except Exception:
            continue

if numeric_field_id is None:
    print('No numeric field found! EDA will be limited.')
else:
    print(f"Using numeric field: {numeric_field_id}")
    # Choose a threshold for filtering
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 5):")
    display(filtered_df.head())

    # Normalize the numeric field in the filtered data
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records (top 5):")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Attempt to find a group field (categorical field)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < 20 and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Grouped mean:")
        display(grouped_df)
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the numeric field and, if possible, the grouped means.

In [ ]:
# Plot histogram or grouped means as available
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded Mass Spectrometry EV dataset using `mlcroissant` and Croissant schema.
- We enumerated record sets and field `@id`s for reference.
- We demonstrated filtering, normalization and visualization of a numeric field.
- Please adapt field `@id`s and parameters to suit your specific exploration! For further insights, consult Croissant metadata (`dataset.metadata`) and schema documentation.